- [论文](https://arxiv.org/abs/2307.06281)
- [GitHub](https://github.com/open-compass/VLMEvalKit)

<!-- # MMBench: Is Your Multi-modal Model an All-around Player? -->
# MMBench：你的多模态模型是全能型吗？

## 摘要

大视觉语言模型（VLMs）近期取得显著进展，展现出强大的多模态感知与推理能力。然而，对这类模型的有效评估仍是一大挑战，严重阻碍了该领域的后续发展。
传统基准（如VQAv2、COCO Caption）虽能提供定量性能指标，但缺乏细粒度能力评估与鲁棒的评价指标；主观基准（如OwlEval）通过人工标注实现模型能力的全面评估，却存在可扩展性差、偏差显著的问题。
针对上述问题，本文提出**双语多模态基准MMBench**，用于评估VLMs的多模态能力，其核心优势包括：
1.  经严格质量控制构建，在评估问题的数量、多样性及覆盖能力上超越现有同类基准；
2.  提出严谨的**循环评估（CircularEval）策略**，引入大语言模型将模型的自由形式输出转换为预定义选项，有效提升低指令遵循能力模型的评估准确性；
3.  支持中英双语选择题，实现跨语言场景下VLMs性能的直接对比。

综上，MMBench是一款**系统化设计的客观基准**，可对视觉语言模型进行**鲁棒且全面的评估**。本文期望MMBench能助力研究界更好地评估模型，推动该领域的未来发展。
MMBench的评估代码已集成至[VLMEvalKit](https://github.com/open-compass/VLMEvalKit)。

## 1 引言

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/radar.png" />
    <span style="font-size: 12px; color: black;">图1</strong>：MMBench 测试定义的 20 个能力维度中 8 个代表性大型视觉语言模型（VLMs）的结果。</span>
</div>

大语言模型（LLMs）近期发展显著，以OpenAI的ChatGPT和GPT-4[OpenAI2023GPT4TR]()为代表的模型，其推理能力已媲美甚至超越人类。受此启发，大视觉语言模型（LVLMs）也迎来变革，GPT-4v[OpenAI2023GPT4TR]()、Gemini-Pro-V[team2023gemini]()和LLaVA[liu2023visual]()等模型在视觉-语言领域的图像内容识别与推理能力上表现突出，远超早期工作。

然而，多数早期研究[gong2023multimodalgpt,zhu2023minigpt,liu2023visual]()仅侧重展示定性示例，缺乏全面的定量实验评估，这为模型间的对比带来极大挑战。近期研究主要采用两种定量评估方式：一是基于现有公共数据集[vqav2,coco_caption]()的客观评估，二是基于人工标注[ye2023mplug,Xu2023LVLMeHubAC]()的主观评估，但两种方式均存在固有缺陷。

VQAv2[vqav2]()、COCO Caption[coco_caption]()、GQA[hudson2019gqa]()和OK-VQA[ok-vqa]()等公共数据集长期用于VLMs的定量评估，可提供准确率、BLEU、CIDEr等**客观指标**。但用于评估先进LVLMs时，这些基准存在两大问题：
1.  **假阴性问题**：现有指标多要求预测结果与参考标准答案完全匹配，例如VQA任务中预测“bicycle”而参考答案为“bike”时会被判定为错误，导致大量假阴性样本；
2.  **缺乏细粒度分析**：现有数据集仅聚焦特定任务的性能评估，无法反映模型的细粒度能力，难以为后续优化提供有效指导。

针对上述问题，OwlEval[ye2023mplug]()和LVLM-eHub[Xu2023LVLMeHubAC]()等研究提出了引入人工的**主观评估策略**。OwlEval基于公共数据集图像构建82个开放式问题，由人工评估模型预测质量；LVLM-eHub受FastChat[zheng2023judging]()启发，搭建在线平台让人工对比两个模型针对同一图像问题的回答。主观评估具备两大优势：
-  **精准匹配**：人类可识别不同表述的同义预测，避免假阴性；
-  **全面评估**：人类可从多维度对比预测结果，最终得分取多能力维度的均值，实现模型的整体评估。

尽管主观评估更全面，但也带来新挑战：
1.  **结果不可复现**：人工评估存在固有偏差，更换标注者可能导致结果不同；
2.  **可扩展性差**：每次实验后都需雇佣标注者，成本高昂；小样本评估数据集易导致统计不稳定，扩大样本则需投入大量人力。

针对传统客观与主观基准的双重缺陷，本文提出**MMBench**——一款系统化设计的客观评估基准，用于鲁棒地评估大视觉语言模型的各项能力。目前，MMBench包含3000余道选择题，覆盖目标定位、社会推理等20个能力维度，每个维度包含至少125道题，题量分布均衡，确保评估的全面性与平衡性。

考虑到部分现有VLMs的指令遵循能力较弱，无法直接输出MMBench题目对应的选项标签（A、B、C等），基于完全匹配的评估方式难以得到准确合理的结论。为减少答案匹配过程中的假阴性样本，本文采用GPT-4将模型的自由形式预测结果匹配到选择题选项中，并输出对应的选项标签。对比实验表明，GPT-4的匹配结果与人工评估的一致性达91.5%，验证了其作为选项提取器的良好一致性与鲁棒性。

为进一步提升评估的鲁棒性，本文提出一种新型评估策略——**循环评估（CircularEval）**（详见[eval_strategy节]()）。基于MMBench，本文对多个知名视觉语言模型（涵盖不同架构与规模）进行了全面评估，并报告了它们在各能力维度的表现。模型的性能排名不仅支持直接对比，也为后续优化提供了宝贵参考。


综上，本文的核心贡献有三：
1. **系统化构建的数据集**：为全面评估VLMs能力，精心构建包含3217道精选问题的数据集，覆盖20项细粒度能力；
2. **鲁棒的评估方法**：提出创新的循环评估策略（CircularEval）提升评估鲁棒性，同时利用GPT-4实现模型预测与选项的匹配，即使对指令遵循能力较弱的VLMs也能有效提取选项；
3. **全面的分析与发现**：基于MMBench对一系列知名视觉语言模型进行全面评估，评估结果可为研究界的后续优化提供重要参考。

<!-- 1.  **研究背景**
    - LLM发展推动LVLM变革，GPT-4v、LLaVA等模型性能显著提升，但早期研究缺乏定量评估，模型对比困难；
    - 现有评估方式的缺陷：客观基准（公共数据集）存在假阴性和细粒度分析不足问题；主观基准（人工评估）存在结果不可复现和可扩展性差问题。
2.  **核心解决方案**：提出客观评估基准MMBench，解决现有评估的双重痛点。
3.  **MMBench关键设计**
    - 数据集：3217道选择题，覆盖20个能力维度，题量分布均衡；
    - 评估优化：用GPT-4匹配模型预测与选项（与人工一致性91.5%），解决低指令遵循能力模型的评估问题；
    - 创新策略：提出CircularEval策略，提升评估鲁棒性。
4.  **实验与价值**：对多架构、多规模VLMs进行全面评估，性能排名支持模型对比，为后续优化提供参考。
5.  **三大核心贡献**：系统化数据集、鲁棒评估方法、全面分析与发现。 -->

## 2 相关工作

## 3 MMBench的构建

MMBench 与现有多模态理解基准的核心差异体现在三方面：
1.  整合多来源图像与问题，基于**层级能力分类体系**评估模型的多样化能力；
2.  执行严格的质量控制流程，保障测试样本的正确性与有效性；
3.  支持**双语（英/中）** 评估，实现视觉-语言模型（VLM）在两种语言环境下的直接性能对比。


<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/MMBench_dist.png" />
    <span style=" font-size: 12px; color: black;">图2</strong>：MMBench 中的能力维度。 目前，MMBench 包含三个能力维度，涵盖 20 种不同的叶子能力。</span>
</div>


### 3.1 MMBench 的层级能力分类法
人类的感知与推理能力是复杂认知的基础，因此 MMBench 将**感知（Perception）** 和**推理（Reasoning）** 作为一级（L-1）能力。在此基础上，进一步划分出 6 个二级（L-2）和 20 个三级（L-3）细粒度能力维度（见图1）。

### 3.2 数据采集与质量控制
1.  **问题采集**
    MMBench 为每个 L-3 能力构建**多选式视觉-语言问答（QA）** 样本，单个样本 $P_i$ 由四元组 $(Q_i, C_i, I_i, A_i)$ 构成，分别对应问题、选项集合（2-4 个选项）、关联图像和正确答案。
    数据由跨学科本硕志愿者基于 10-20 个示例题扩展采集，80% 以上样本来自互联网，其余 20% 的图像取自公共数据集验证集（非训练集），问题为人工构建。

2.  **质量控制**
    原始数据存在两类低质量样本：
    - 仅通过文本即可推理答案（无法评估多模态能力）；
    - 样本本身存在错误（问题、选项或答案有误）。

    对应过滤策略：
    - 针对**文本可解样本**：采用大语言模型（LLM，如 GPT-4、Gemini-Pro）的**多数投票**机制，若超过半数 LLM 仅通过文本答对，则人工验证后剔除；
    - 针对**错误样本**：采用视觉-语言模型（VLM）的**全错筛选**机制，若所有主流 VLM 均答错，则人工验证后剔除。

3.  **MMBench-CN（中文版本）**
    基于 GPT-4 将 MMBench 的问题和选项翻译为中文（专有名词、符号、代码除外），并经人工验证有效性，支持 VLM 在英/中语境下的直接性能对比。

### 3.3 MMBench 统计信息
1.  **数据统计**：共 3,217 个样本，覆盖 20 个 L-3 能力。为保证评估的均衡性，每个 L-3 能力至少包含 125 个样本。
2.  **数据划分**：按 4:6 比例划分为 **dev 集**和 **test 集**。dev 集完全公开（含标准答案）；test 集仅公开样本，标准答案需通过提交预测结果至评估服务器获取。

<!-- ## 关键术语对照表
| 英文术语 | 中文翻译 |
|----------|----------|
| Hierarchical Ability Taxonomy | 层级能力分类体系 |
| Perception / Reasoning | 感知 / 推理 |
| Vision-Language Model (VLM) | 视觉-语言模型 |
| Majority Voting | 多数投票 |
| Text-only Sample | 文本可解样本 |
| dev / test subset | 开发集 / 测试集 | -->

## 4 评估策略

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/ABCD.png" />
    <span style=" font-size: 12px; color: black;">图4：样本 VLM（所有 CircularEval 记录）的基准答案和预测选择分布。</strong>由于 MMBench 中存在只有 2/3 选项的问题，真实选择的分布并不完全均匀。</span>
</div>

MMBench 提出两种核心策略以实现**低成本、高鲁棒性**的模型评估：
1.  **LLM 辅助选项提取**：针对 VLM 的自由格式输出，利用先进大语言模型（LLM，如 GPT-4）提取选项标签，解决规则匹配失效问题；
2.  **CircularEval 循环评估**：通过多次打乱选项顺序测试同一问题，要求模型全部答对才算通过，有效放大模型间的性能差距。

### 4.1 LLM 使用LLM的选项提取
VLM 的指令遵循能力差异显著：部分模型可直接输出标准选项标签（如 A、B），但多数闭源模型（如 GPT-4v、Qwen-VL-Max）和未经过多选题训练的开源模型，会输出自由格式文本（如选项内容而非标签）。
为此，设计**两阶段通用提取流程**：
1.  **阶段 1：启发式匹配**
    优先尝试从 VLM 输出中提取选项标签（A/B/C/D）。若成功，直接作为模型预测结果；若失败，进入阶段 2。
2.  **阶段 2：LLM 辅助匹配**
    将**问题、选项、VLM 自由输出**输入 LLM（默认使用 gpt-4-0125），要求其将输出与选项对齐并返回标签。若输出与所有选项均不匹配，返回伪标签 `Z`。
    实验表明，LLM 几乎能处理所有启发式匹配失效的案例，匹配结果与真实标签一致则判定样本正确。

### 4.2 LLM 作为选项提取器：可行性分析
1.  **VLM 指令遵循（IF）能力差异大**
    先导实验显示：部分 VLM（如 LLaVA 系列、CogVLM-Chat）的启发式匹配成功率超 99%，但闭源模型和大量开源模型表现较差（如 VisualGLM-6B 仅 64.8%）。
    对低匹配率模型，LLM 辅助提取可**显著提升最终准确率**（如 VisualGLM-6B 准确率提升 23.2%）。
    关键发现：**指令遵循能力与多模态理解能力无必然关联**。例如，OpenFlamingo v2 指令遵循能力顶尖，但 MMBench 整体性能垫底。



<div style="width: 80%; margin: auto; font-family: sans-serif;">
    <div style="background-color: white; padding: 20px; display: grid; 
        grid-template-columns: 1fr 1fr; 
        justify-items: center;
        align-items: center;">
        <img src="./assets/statistics_of_if_capabilities.png"/>
        <img src="./assets/Alignment.png" />
        <div style="font-size: 13px; color: #333; text-align: center;">
            <strong>表1：VLM 的中频能力统计数据。</strong>： 我们报告了 VLMs 的启发式匹配成功率，以及基于 LLM 的选择提取前后的准确性。 在“X+Y”中，X 表示基于匹配的准确性，Y 表示使用 LLM 作为选择提取器的收益。
        </div>
        <div style="font-size: 13px; color: #333; text-align: center;">
            <strong>图5：人类与不同大型语言模型之间的比对率。</strong>“ChatGPT”就是“GPT-3.5-Turbo”。开源的大型语言模型是“聊天”变体。
        </div>
    </div>
    <div style="background-color: white; padding: 20px; text-align: center;">
        <img src="./assets/circular_eval.png" style="width: 100%; height: auto; border-radius: 4px;" />
        <div style="font-size: 14px; color: #333; margin-top: 15px;">
            <strong>图6： 循环评估策略。</strong>在 CircularEval 中，一个问题会多次测试，选择循环平移，VLM 需要在所有测试中都成功。在这个例子中，VLM 在第三次通过时失败，因此被视为问题失败。
        </div>
    </div>
</div>

2.  **LLM 提取的质量与稳定性**
    为验证 LLM 提取效果，从启发式匹配失效的样本中随机选取 10%（约 420 个），由志愿者人工标注选项标签，计算 LLM 与人工标注的**对齐率**：
    - 闭源 LLM：GPT-4 对齐率最高（91.5%），GPT-3.5-Turbo 和 Qwen-Max 约 85%；
    - 开源 LLM：性能差异较大，InternLM2-7B 表现最优（87%），超越 GPT-3.5-Turbo。
    后续实验默认使用 **gpt-4-0125** 作为提取器，且顶尖 LLM 间的对齐率差异对 VLM 最终性能影响极小。

### 4.3 CircularEval 循环评估策略
多选题格式存在固有缺陷：随机猜测对 4 选项题可获得 $\sim25\%$ 的 Top-1 准确率，且 VLM 存在**选项偏好**（倾向于预测某一固定标签），均会降低评估区分度。
为此，提出 **CircularEval 策略**：
1.  对每个问题，按**选项数量 $N$** 进行 $N$ 次测试，每次循环移位打乱选项与答案的顺序，生成新提示；
2.  模型需**在所有测试轮次中均答对**，才算成功解决该问题；
3.  实际实现中，若模型某一轮答错，可提前终止测试，因此实际成本低于 $N$ 倍单轮测试。
该策略在**鲁棒性与成本间实现良好平衡**，能更有效地区分不同 VLM 的性能差异。


<!-- | 英文术语 | 中文翻译 |
|----------|----------|
| LLM-involved Choice Extraction | LLM 辅助选项提取 |
| Heuristic Matching | 启发式匹配 |
| Instruction Following (IF) Capability | 指令遵循能力 |
| Alignment Rate | 对齐率 |
| CircularEval (Circular Evaluation) | 循环评估 |
| Circular Shifting | 循环移位 | -->

## 5 评估结果

==TODO补充图表==

### 5.1 实验设置
在MMBench基准上，对**三大类模型**开展零样本（zero-shot）评估，所有模型使用相同提示词，通过开放式生成获取预测结果，并以`gpt-4-0125`作为选项提取器。
- 纯文本模型：GPT-4
- 开源多模态大模型（VLMs）：OpenFlamingo、MiniGPT4、InstructBLIP、LLaVA系列、IDEFICS、CogVLM、Qwen-VL、Yi-VL等
- 闭源多模态大模型：Qwen-VL-[Plus/Max]、Gemini-Pro-V、GPT-4v

所有实验基于**VLMEvalKit**完成，附录提供了开源模型的架构、参数量及更多实验设置下的补充结果。

### 5.2 核心结果

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/circular_vs_vanilla.png" />
    <span style=" font-size: 12px; color: black;"><strong>表2：循环评估 vs. 普通评估（VanillaEval）。</strong> 我们报告了 MMBench-dev 上所有 VLM 的 CircularEval 前一准确率和准确率下降（与 VanillaEval 相比）。</span>
</div>

__循环评估（CircularEval）vs. 普通评估（VanillaEval）__
- 核心差异：VanillaEval为单轮推理，CircularEval要求多轮推理且结果一致，是更严格的评估范式。
- 关键发现：
  1. 切换至CircularEval后，**绝大多数模型准确率显著下降**（如OpenFlamingo v2从36.7%降至2.6%，闭源模型如GPT-4v也下降约10%）。
  2. CircularEval能放大不同模型间的性能差距（如LLaVA-v1.5-13B与7B的差距从2.1%扩大至4.7%）。
  3. 后续实验均采用**CircularEval**作为默认评估方式。

在MMBench测试集上，评估了模型的**整体性能**及6类L-2能力：
- 粗感知（CP）、细粒度感知（单实例FP-S/跨实例FP-C）、属性推理（AR）、逻辑推理（LR）、关系推理（RR）

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/mmbench_l2.png" />
    <span style=" font-size: 12px; color: black;"><strong>表 3：MMBench 测试集（L-2 能力）的循环评估结果。</strong> 采用缩写：逻辑推理的 LR;AR 代表属性推理;RR 代表关系推理;FP-C 代表细粒度感知（交叉实例）;FP-S 代表细粒度感知（单一实例）;CP 代表粗糙感知。模型按整体准确率（组内）的升序排序。 标注为*的开源模型在模型训练中整合了内部数据。</span>
</div>

__MMBench测试的表现__:
1. 纯文本模型（GPT-4-Turbo）性能接近随机水平（整体准确率2.9%）。
2. 开源模型中，**InternLM-XComposer2性能最优**（整体78.1%），大幅领先其他开源/部分闭源模型。
3. LLaVA系列（含Yi-VL）基于相同架构，展现出强劲性能，仅次于顶尖闭源模型（GPT-4v、Qwen-VL-Max）。
4. 小参数量模型（如MiniCPM-V，≤3B）实现了超60%的准确率，验证了小模型的潜力。
5. MiniGPT、IDEFICS等模型性能显著落后，OpenFlamingo v2因缺乏指令微调，性能接近随机。

__大语言模型（LLM）是影响VLM性能的关键因素__
1. 同架构模型中，更换更优LLM可实现全能力维度提升（如LLaVA系列从Vicuna-v1.5更换为InternLM2-20B后，推理任务性能提升尤为显著）。
2. 同系列LLM中，参数量扩增可提升性能（如7B升级至13B后，MiniGPT4、InstructBLIP、LLaVA v1.5的整体准确率分别提升8.3%、1.5%、3.5%）。

__MMBench-CN上的表现__:
- 多数模型在MMBench-CN上性能低于MMBench，差异源于预训练/指令微调阶段的**英中文本分布不均衡**。
- 顶尖模型（如InternLM-XComposer2）的中英性能差距极小（＜1%），优势源于**双语能力更强的LLM基座**或**平衡的跨语言多模态语料**。
- 例外情况：OpenFlamingo v2、VisualGLM、Qwen-VL-Plus在MMBench-CN上性能未下降。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/envscn-crop.png" />
    <span style=" font-size: 12px; color: black;"><strong>图 7：MMBench 和 MMBench-CN 测试分段的性能。</strong>模型按平均表现的递增排序。ILM 代表 InternLM。</span>
</div>

### 5.3 细粒度分析
__闭源模型的内容审核影响__
- 所有闭源模型均存在显式内容审核，GPT-4v、Gemini-Pro-V、Qwen-VL-Max的拒绝回答率分别为1.8%、1.6%、0.1%（GPT-4v的拒绝问题中74%与名人识别相关）。
- 假设拒绝问题均能完美回答，计算**性能上限**：审核对准确率的最大影响为2.4%，整体影响不显著。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; display: grid; 
    grid-template-columns: 1fr 1fr; /* 两列均等 */ 
    grid-template-rows: auto auto; /* 第一行图片，第二行标题 */ 
    gap: 10px 20px; /* 行间距 10px，列间距 20px */ 
align-items: center; justify-items: center;">
    <img src="./assets/upperbound.png" style="max-width: 100%; height: auto;" />
    <img src="./assets/API_rejection.png" style="max-width: 100%; height: auto;" />
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;">
        <strong>表4</strong>：闭源 VLM 的“上界”计价估计。
    </div>
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;">
        <strong>图8</strong>：闭源 VLM 的内容审核案例。
    </div>
</div>

__闭源vs.开源：性能差距何在？__
闭源模型整体性能更稳定，与开源顶尖模型（LLaVA-InternLM2-20B）的细粒度对比显示：
- 闭源模型的核心优势场景：
  1. 结构化图文理解（如代码、表格、图表、布局）
  2. 需外部知识的任务（如名人识别、物理属性推理、自然关系推理）
- 闭源模型在**其他感知/推理任务**上无明显优势。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/API_fine.png" />
    <span style=" font-size: 12px; color: black;"><strong>图 9</strong>闭源 VLM 与开源 VLM 在细致层面上的比较。</span>
</div>

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/HardExample-crop.png" />
    <span style=" font-size: 12px; color: black;"><strong>图 10</strong>难点例子属于 4 个 L-3 技能中 $A_{max}$。 所有 VLM 对 CircularEval 下可视化的示例做出了错误的预测。</span>
</div>

__MMBench中的难题（Hard Cases）__
通过分析各L-3能力的**最大准确率（$A_{max}$）**，发现现有模型的共性短板：
1. 视觉底层特征识别能力弱（无法准确判断图像亮度、清晰度、对比度、伪影）
2. 结构化视觉输入理解困难（即使简单表格/图表也存在障碍）
3. 物体间空间关系推理能力不足（2D/3D空间均存在缺陷）

## 6 结论

我们提出多模态基准测试集MMBench，包含 3000 余道选择题，覆盖 20 个能力维度，用于对多模态大模型（VLM）进行客观评估。为获取鲁棒、可靠的评估结果，我们设计了全新评估策略CircularEval—— 该策略比传统单轮推理评估更严格，且成本可控。针对部分 VLM 指令跟随能力不足的问题，我们额外引入大语言模型（LLM）从模型预测结果中提取答案选项，以提升评估准确性。我们在 MMBench 上对 20 余种主流 VLM（覆盖不同架构与参数量级）开展全面评估，所得结果为模型的未来优化提供了重要参考。